# Tools and Agents in LangChain

# What You Will Learn

In this notebook, you will learn:

- What Tools are in LangChain
- What Agents are in LangChain
- Difference between Tools and Agents
- How Agents decide which tool to use
- How to use Wikipedia Tool
- How to use Arxiv Tool
- How tool calling works in LangChain
- How agents perform reasoning and actions
- How to create agents using modern LangChain 1.x
- How to stream and observe agent execution
- How LLMs interact with external tools



## Tool Definition (Interview Definition)

A Tool in LangChain is a callable function that allows an LLM to interact with external systems and perform specific tasks such as web search, API calls, database queries, calculations, or code execution.

Tools have well-defined:
- inputs
- outputs
- functionality

The language model decides:
- when to use a tool
- which tool to use
- what arguments to pass

Examples:
- Web Search Tool
- Wikipedia Tool
- Arxiv Research Tool
- Calculator Tool

---

# Agent Definition (Interview Definition)

An Agent in LangChain is a system that combines a language model with tools to enable reasoning, decision-making, and multi-step task execution.

An agent can:
1. understand the user request
2. decide which tools are needed
3. execute tools
4. observe outputs
5. solve complex tasks step-by-step until the final response is generated

Unlike normal chains:

```text
Input → Prompt → LLM → Output
```

Agents dynamically decide actions during runtime.

---

# Example

User Question:

```text
"What is the latest research paper on RAG say?"
```

## Agent Workflow

```text
User Query
   ↓
Agent Thinks
   ↓
Chooses Arxiv Tool
   ↓
Retrieves Research Paper
   ↓
Observes Output
   ↓
Summarizes Information
   ↓
Generates Final Response
```

---

# Example Final Response

```text
The latest research paper on Retrieval-Augmented Generation (RAG)
introduces improved retrieval techniques for enhancing factual accuracy
and reducing hallucinations in large language models.
```

---

# How an Agent Works

```text
User Query
    ↓
Agent Thinks
    ↓
Chooses Tool
    ↓
Executes Tool
    ↓
Observes Result
    ↓
Repeats if Needed
    ↓
Final Response
```

The agent continues this process until:
- the final answer is generated
- or an iteration limit is reached

---

# Summary

| Component | Role |
|---|---|
| Tool | Performs a specific task |
| Agent | Decides which tools to use and manages the workflow |

---

# Why Agents are Powerful

Agents allow LLMs to:
- access real-time information
- interact with external systems
- perform multi-step reasoning
- solve complex workflows
- dynamically choose actions

## Wrapper vs QueryRun

## Wrapper

A Wrapper is a helper that communicates with external APIs or websites like Arxiv or Wikipedia and fetches the data.

Think:
"Go and get the information."

Example:
- `ArxivAPIWrapper`
- `WikipediaAPIWrapper`

---

## QueryRun

A QueryRun is a LangChain Tool that uses the wrapper to process user queries and return results.

Think:
"Ask a question and get the answer."

Example:
- `ArxivQueryRun`
- `WikipediaQueryRun`

---

## Real-Life Example

- Wrapper → goes to Arxiv and fetches research papers
- QueryRun → asks: "What are the latest ML papers?" and returns the results using the wrapper

---

## Summary

| Component | Purpose |
|---|---|
| Wrapper | Fetches data from APIs/websites |
| QueryRun | Uses the wrapper to answer user queries |

In [1]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper

In [2]:
## Used the inbuilt tool of arxiv
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1 , doc_content_chars_max=2500)
arxiv_tool=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv_tool.name

'arxiv'

In [3]:
## Used the inbuilt tool of wikipedia
api_wrapper_wikipedia=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki_tool=WikipediaQueryRun(api_wrapper=api_wrapper_wikipedia)
wiki_tool.name

'wikipedia'

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv
load_dotenv()


USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [5]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [6]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
Docs=loader.load()
Documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=250).split_documents(Docs)
vectorstore = FAISS.from_documents(Documents,HuggingFaceEmbeddings())
retriever=vectorstore.as_retriever()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(retriever=retriever,name="langsmith-search",description="search anything about Langsmith")

In [8]:
retriever_tool.name

'langsmith-search'

In [9]:
retriever_tool.description

'search anything about Langsmith'

In [10]:
tools=[arxiv_tool,wiki_tool,retriever_tool]

In [11]:
tools

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=2500)),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\shaik\\Music\\Udemy\\P-GenAI\\9-langchain-agents-and-tools\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='search anything about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000023B133CF9C0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000023B66DCEAC0>)]

In [12]:
tools= [wiki_tool,arxiv_tool,retriever_tool]
print(tools[0].name, tools[1].name, tools[2].name)


wikipedia arxiv langsmith-search


In [13]:
from langchain_groq import ChatGroq
load_dotenv()
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=GROQ_API_KEY,model="openai/gpt-oss-120b")

In [15]:
from langchain.agents import create_agent


agent= create_agent(model=llm,tools=tools, system_prompt="You are a helpful assistant that answers questions accurately.")



In [22]:
response = agent.invoke(
    {
        "messages": [
            ("user", "What is LangChain?")
        ]
    }
)

response["messages"][-1].content

'**LangChain** is an open‑source software framework designed to make it easy to build applications that use large language models (LLMs) such as GPT‑4, Claude, or Llama\u202f2.  \n\n### Core idea\nInstead of treating an LLM as a single “black‑box” API call, LangChain provides a set of building blocks that let developers **compose** LLM calls into more complex, reliable workflows—called *chains*—and integrate them with external data sources, tools, and memory.\n\n### Main components  \n\n| Component | What it does | Typical use |\n|-----------|--------------|-------------|\n| **LLM wrappers** | Uniform interfaces to many LLM providers (OpenAI, Anthropic, Cohere, etc.) | Swap models without changing code |\n| **Prompt templates** | Parameterized prompts with variables, partial rendering, and versioning | Reuse and manage prompts at scale |\n| **Chains** | Sequential or conditional pipelines of LLM calls, prompts, and other functions | Build multi‑step reasoning, summarization, Q&A |\n| *

In [30]:
response = agent.stream(
    {
        "messages": [
            ("user", "What is langchain?")
        ]
    },
    stream_mode="values"
)

for step in response:
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is langchain?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (fc_7a6594b3-deb5-4a3b-a174-3c162020a9ba)
 Call ID: fc_7a6594b3-deb5-4a3b-a174-3c162020a9ba
  Args:
    query: LangChain
================================= Tool Message =================================
Name: wikipedia

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of languag
================================== Ai Message ==================================

**LangChain** is an open‑source software framework that makes it easier to build applications powered by large language models (LLMs).  

### Core idea
Instead of treating an LLM as a standalone “black box,” LangChain provides a set of reusab

In [33]:
response = agent.stream(
    {
        "messages": [
            ("user", "What's the paper 1706.03762 about?")
        ]
    },
    stream_mode="values"
)

for step in response:
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the paper 1706.03762 about?
================================== Ai Message ==================================
Tool Calls:
  arxiv (fc_bb0fd4ed-20e6-4dd2-83c7-c4d5b2176793)
 Call ID: fc_bb0fd4ed-20e6-4dd2-83c7-c4d5b2176793
  Args:
    query: 1706.03762
================================= Tool Message =================================
Name: arxiv

Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and c

In [38]:
response = agent.stream(
    {
        "messages": [
            ("user", "Search latest research paper on RAG")
        ]
    },
    stream_mode="values"
)

for step in response:
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Search latest research paper on RAG
================================== Ai Message ==================================
Tool Calls:
  arxiv (fc_12611f3f-be3d-488a-9b9d-0eeb10376b8e)
 Call ID: fc_12611f3f-be3d-488a-9b9d-0eeb10376b8e
  Args:
    query: retrieval-augmented generation
================================= Tool Message =================================
Name: arxiv

Published: 2025-06-14
Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
Authors: Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang
Summary: We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each 